In [0]:
%run ../00-common/01.enviroment-config

In [0]:
bronze_table = f'{catalog_name}.{bronze_schema}.constructors'
silver_table = f'{catalog_name}.{silver_schema}.constructors'

In [0]:
constructors_df = spark.read.table(bronze_table)

In [0]:
import pyspark.sql.functions as F

In [0]:
constructors_selected_df = constructors_df.select(
    F.col("constructorId"),
    F.col("name"),
    F.col("nationality"),
    F.col("ingestion_timestamp"),
    F.col("source_file")
)

In [0]:
constructors_renamed_df = (
    constructors_selected_df
        .withColumnRenamed("constructorId", "constructor_id")
        .withColumnRenamed("name", "constructor_name")
)

In [0]:
constructors_valid_df = constructors_renamed_df.filter(
    F.col("constructor_id").isNotNull()
)

In [0]:
constructors_distinct_df = constructors_valid_df.dropDuplicates(["constructor_id"])

In [0]:
constructors_final_df = (
    constructors_distinct_df
        .withColumn("nationality", F.initcap(F.col("nationality")))
)

In [0]:
display(constructors_final_df)

In [0]:
(
    constructors_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
)